# Cinemática Directa — MyCobot 280

**Problemas P1 + P2**
**Curso:** ROB 2026

## Contenido del notebook

1. Importaciones
2. Parámetros DH del MyCobot 280
3. 5 configuraciones de prueba para P2
4. Clase `ForwardKinematics` (matriz DH, FK completa, espacio de trabajo)
5. Cálculo de FK para las 5 configuraciones
6. Generación y graficación del espacio de trabajo XZ


## 1. Importaciones

Se usan `math` y `numpy` para los cálculos matriciales, y `matplotlib` para graficar el espacio de trabajo.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

## 2. Parámetros DH del MyCobot 280

Tabla DH medida físicamente sobre el robot real.
Cada fila es `(a, d, alpha)` para cada articulación J1–J6.

| # | a (mm) | d (mm) | alpha |
|---|---|---|---|
| J1 | 0 | 0 | 0 |
| J2 | 0 | 134.65 | +90° |
| J3 | −110 | 0 | 0 |
| J4 | −96 | 0 | 0 |
| J5 | 0 | 63.4 | −90° |
| J6 | 50 | 75.05 | +90° |


In [ ]:
# Parámetros DH: (a, d, alpha)
PARAMETROS_DH = [
    (0.0,    0.0,    0.0),
    (0.0,  134.65,   math.pi/2),
    (-110.0, 0.0,    0.0),
    (-96.0,  0.0,    0.0),
    (0.0,   63.4,   -math.pi/2),
    (50.0,  75.05,   math.pi/2),
]

## 3. Cinco Configuraciones para P2

Conjunto de 5 vectores de ángulos articulares que se usarán para calcular FK y comparar contra `mc.get_coords()` del robot real.


In [ ]:
# 5 CONFIGURACIONES PARA P2
CONFIGURACIONES = [
    [0,    0,    0,   0,   0, -45],
    [0,  -45,   45,   0,   0, -45],
    [45, -30,   60,   0,  30, -45],
    [-30,-60,   30,   0, -30, -45],
    [90, -45,   45,   0,   0, -45],
]

## 4. Clase `ForwardKinematics`

Esta clase implementa:

- `matriz_dh(a, d, alpha, theta)` — construye la matriz de transformación homogénea 4×4 para una articulación.
- `calcular(angulos_grados)` — multiplica T₁·T₂·…·T₆ y devuelve la matriz T₀⁶ completa.
- `posicion_extremo(angulos_grados)` — extrae la posición XYZ del extremo desde T₀⁶.
- `espacio_trabajo(muestras)` — genera una nube de puntos en el plano XZ muestreando ángulos aleatorios dentro de los rangos físicos del robot.


In [ ]:
class ForwardKinematics:

    def __init__(self):
        self.parametros_dh = PARAMETROS_DH

    # ---------- MATRIZ DH ----------
    def matriz_dh(self, a, d, alpha, theta):
        ct = math.cos(theta)
        st = math.sin(theta)
        ca = math.cos(alpha)
        sa = math.sin(alpha)

        T = np.array([
            [ct, -st*ca,  st*sa, a*ct],
            [st,  ct*ca, -ct*sa, a*st],
            [0,      sa,     ca,    d],
            [0,       0,      0,    1]
        ])
        return T

    # ---------- FK COMPLETA ----------
    def calcular(self, angulos_grados):
        if len(angulos_grados) != 6:
            raise ValueError("Se necesitan 6 ángulos")

        T_total = np.eye(4)

        for i in range(6):
            a, d, alpha = self.parametros_dh[i]
            theta = math.radians(angulos_grados[i])
            T = self.matriz_dh(a, d, alpha, theta)
            T_total = T_total @ T

        return T_total

    # ---------- POSICIÓN FINAL XYZ ----------
    def posicion_extremo(self, angulos_grados):
        T = self.calcular(angulos_grados)
        return T[:3, 3]

    # ---------- ESPACIO DE TRABAJO XZ ----------
    def espacio_trabajo(self, muestras=5000):
        rng = np.random.default_rng()
        puntos = []

        for _ in range(muestras):
            angulos = [
                rng.uniform(-180, 180),
                rng.uniform(-135, 135),
                rng.uniform(-150, 150),
                rng.uniform(-145, 145),
                rng.uniform(-165, 165),
                rng.uniform(-180, 180),
            ]
            pos = self.posicion_extremo(angulos)
            puntos.append([pos[0], pos[2]])

        return np.array(puntos)

## 5. Cálculo de FK para las 5 Configuraciones

Se instancia la clase `ForwardKinematics` y se calcula la matriz T₀⁶ y la posición XYZ del extremo para cada una de las 5 configuraciones definidas arriba.


In [ ]:
fk = ForwardKinematics()

print("="*60)
print("P1 + P2 — Cinemática Directa MyCobot 280")
print("="*60)

for i, config in enumerate(CONFIGURACIONES, 1):

    T = fk.calcular(config)
    pos = fk.posicion_extremo(config)

    print(f"\nConfiguración C{i}")
    print("Joints:", config)
    print("\nMatriz T0_6:")
    print(np.round(T, 2))

    print("\nPosición XYZ:")
    print(f"X = {pos[0]:.2f} mm")
    print(f"Y = {pos[1]:.2f} mm")
    print(f"Z = {pos[2]:.2f} mm")

## 6. Espacio de Trabajo Alcanzable (plano XZ)

Se generan 8000 muestras aleatorias dentro de los rangos articulares del MyCobot 280 y se proyectan en el plano XZ.
Esto permite visualizar la región accesible del robot.

El resultado se guarda como `espacio_trabajo_XZ.png`.


In [ ]:
print("\nGenerando nube XZ...")

puntos = fk.espacio_trabajo(8000)

plt.figure(figsize=(8,7))

plt.scatter(
    puntos[:,0],
    puntos[:,1],
    s=1,
    alpha=0.3
)

plt.xlabel("X (mm)")
plt.ylabel("Z (mm)")
plt.title("Espacio de Trabajo MyCobot 280 — Plano XZ")
plt.grid(True)
plt.axis("equal")

plt.savefig("espacio_trabajo_XZ.png", dpi=150)

print("\n[OK] Imagen guardada: espacio_trabajo_XZ.png")

plt.show()